In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/term-deposit-marketing-2020.csv")

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numerical_cols = [
    "age",
    "balance",
    "day",
    "duration",
    "campaign"
]

categorical_cols = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("number", StandardScaler(), numerical_cols),
        ("category", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)



In [4]:
from sklearn.model_selection import train_test_split

X = df.drop("y", axis=1)
y = df["y"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
#setup baseline models

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


models = {
    "Logistic Regression": {
        "pipeline": Pipeline([
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(random_state=42, max_iter=1000))
        ]),
        "params": {
            "model__C": [0.01, 0.1, 1, 10]
        }
    }}

In [6]:
from sklearn.model_selection import RepeatedStratifiedKFold


cv = RepeatedStratifiedKFold(   
    n_splits=5,
    n_repeats=10,
    random_state=42
)

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score


results = []


for name, config in models.items():
    # Grid search
    grid_search = GridSearchCV(config["pipeline"], config["params"], cv=cv, scoring="accuracy", n_jobs=-1)

    grid_search.fit(X_train, y_train)

    # Get best model
    best_model = grid_search.best_estimator_

    # Evaluate best model using RepeatedStratifiedKFold
    scores = cross_val_score(best_model, X_train, y_train, cv=cv, scoring="accuracy")
    

    results.append({
        "Model": name,
        "Best Accuracy": scores.mean(),
        "Best Standard Deviation": scores.std(),
        "Best Parameters": grid_search.best_params_
        
    })


In [ ]:
scores = cross_val_score(best_model, X_test, y_test, cv=cv, scoring="accuracy")

print(scores.mean())
print(scores.std())

#initial test for accuracy, need to delve further due to the imbalance between target options

0.9382375000000001
0.0038010894819775045


In [8]:
results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

              Model  Best Accuracy  Best Standard Deviation  Best Parameters
Logistic Regression       0.933953                 0.001815 {'model__C': 10}
